In [ ]:
df = pd.read_csv('datasets/SMSSpamCollection', sep='\t', names=['label', 'text'])
df

x = df['text']
y = df['label']

# Data Cleaning
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk import pos_tag

def clean_text(sent):
    # tokenize the data
    tokens1 = word_tokenize(sent)
    # remove punctuation and numbers
    tokens2 = [token for token in tokens1 if token.isalpha()]
    # remove stopwords
    tokens3 = [token.lower() for token in tokens2 if token.lower() not in stopwords.words('english')]
    # apply stemming
    ps = PorterStemmer()
    tokens4 = [ps.stem(token) for token in tokens3]
    return tokens4

# Vectorization

from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(analyzer=clean_text)
x_new = tfidf.fit_transform(x)
tfidf.get_feature_names_out()
len(tfidf.get_feature_names_out())

x_train, x_test, y_train, y_test = train_test_split(x_new, y, random_state=0)

In [ ]:
#ANN

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

tokenizer = Tokenizer()
tokenizer.fit_on_texts(lines)

tokenizer.word_docs
tokenizer.word_counts
tokenizer.word_index
tokenizer.index_word
tokenizer.index_docs

mat = tokenizer.texts_to_matrix(lines)

for i in range(len(mat[0])):
    if mat[0][i]>0:
        print(tokenizer.index_word[i])

words_in_first_row = [tokenizer.index_word[i] for i in range(len(mat[0])) if mat[0][i]>0]
words_in_first_row

seq = tokenizer.texts_to_sequences(lines)
seq

padded = pad_sequences(seq, maxlen=max([len(i) for i in seq]), padding='pre')
padded

In [ ]:
import nltk, re, string, os
from nltk.corpus import stopwords
from tensorflow.keras.preprocessing.text import Tokenizer
import numpy as np

# load doc in memory
def load_doc(filename):
    f = open(filename)
    text = f.read()
    f.close()
    return text

load_doc('datasets/review_polarity/txt_sentoken/neg/cv004_12641.txt')

from nltk.tokenize import word_tokenize
swords = stopwords.words('english')

# Function to clean the documents
def clean_doc(doc):
    tokens = word_tokenize(doc)
    tokens1 = [token for token in tokens if token.isalpha()]
    tokens2 = [token for token in tokens1 if token not in swords]
    tokens3 = [token for token in tokens2 if len(token) > 1]
    return tokens3  

d = load_doc('datasets/review_polarity/txt_sentoken/neg/cv004_12641.txt')
clean_doc(d)

# load the file, clean the data and return thr string
def doc_to_line(filename):
    doc = load_doc(filename)
    tokens = clean_doc(doc)
    return ' '.join(tokens)

doc_to_line('datasets/review_polarity/txt_sentoken/neg/cv004_12641.txt')

f = open('datasets/vocab.txt')
vocab = f.read().split()
vocab

# load the file, clean the data and return thr string
def doc_to_line(filename):
    doc = load_doc(filename)
    tokens = clean_doc(doc)
    tokens = [token for token in tokens if token in vocab]
    return ' '.join(tokens)

doc_to_line('datasets/review_polarity/txt_sentoken/neg/cv004_12641.txt')

def process_train(directory):
    documents = []
    for filename in os.listdir(directory):
        if not filename.startswith('cv9'):
            path = directory + '/' + filename
            docs = load_doc(path)
            tokens = clean_doc(docs)
            documents.append(tokens)
    return documents

tr = process_train('datasets/review_polarity/txt_sentoken/neg/')

def process_test(directory):
    documents = []
    for filename in os.listdir(directory):
        if filename.startswith('cv9'):
            path = directory + '/' + filename
            docs = load_doc(path)
            tokens = clean_doc(docs)
            documents.append(tokens)
    return documents

te = process_test('datasets/review_polarity/txt_sentoken/neg/')

def process_docs(directory, is_train):
    documents = []
    for filename in os.listdir(directory):
        if is_train and filename.startswith('cv9'):
            continue
        if not is_train and not filename.startswith('cv9'):
            continue
        path = directory + '/' + filename
        docs = load_doc(path)
        tokens = clean_doc(docs)
        documents.append(tokens)
    return documents

tr = process_docs('datasets/review_polarity/txt_sentoken/neg/', True)
len(tr)
te = process_docs('datasets/review_polarity/txt_sentoken/neg/', False)
len(te)

def load_data(is_train):
    neg = process_docs('datasets/review_polarity/txt_sentoken/neg/', is_train)
    pos = process_docs('datasets/review_polarity/txt_sentoken/pos/', is_train)
    docs = neg + pos

    labels = [0 for i in range(len(neg))] + [1 for i in range(len(pos))]
    return docs,labels

train_data , train_labels = load_data(True)
len(train_data) , len(train_labels)

test_data , test_labels = load_data(False)
len(test_data) , len(test_labels)

# Data Preparation

def create_tokenizer(lines):
    tokenizer = Tokenizer()
    tokenizer.fit_on_texts(lines)
    return tokenizer

tokenizer = create_tokenizer(train_data)

len(tokenizer.word_index)

X_train = tokenizer.texts_to_matrix(train_data)
X_test = tokenizer.texts_to_matrix(test_data)

from keras.models import Sequential
from keras.layers import Input, Dense
from keras.utils import to_categorical, plot_model

# create the model
model = Sequential()

# input layer
model.add(Input(shape=(X_train.shape[1],)))

# hidden layer
model.add(Dense(256 , activation='relu'))
# model.add(Dense(128 , activation='relu'))

# output layer
model.add(Dense(1, activation='sigmoid'))


In [ ]:
#sequence

alphabets = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ'
list(enumerate(alphabets))

int_to_char = dict(enumerate(alphabets))

char_to_int = dict((v,i) for i, v in enumerate(alphabets))
char_to_int

# X and y for model
seq_length = 1
x = []
y = []

for i in range(len(alphabets) - seq_length):
    seq_in = [alphabets[i]]
    seq_out = alphabets[i+1]
    print(seq_in,'-->',seq_out)
    x += [[char_to_int[seq_in[0]]]]
    y += [char_to_int[seq_out]]

import numpy as np
x = np.reshape(x, (25,1,1))
x

x_scaled = x / 25
x_scaled

from keras.utils import to_categorical
y_new = to_categorical(y)
y_new

from keras.layers import SimpleRNN, Input, Dense
from keras.models import Sequential

model = Sequential()

model.add(Input((1,1)))
model.add(Dense(32, activation='relu'))
model.add(SimpleRNN(32))
model.add(Dense(26, activation='softmax'))

model.summary()
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])


#test

new_char = 'X'
new_char = char_to_int[new_char]
new_char = np.reshape(new_char,(1,1,1))
new_char = new_char / 25
pred = model.predict(new_char, verbose=0)
int_to_char[pred.argmax()]


seq_length = 3
x = []
y = []
for i in range(len(alphabets) - seq_length):
    seq_in = alphabets[i:i+seq_length]
    seq_out = alphabets[i+seq_length]
    print(seq_in,'-->',seq_out)
    x.append([char_to_int[char] for char in seq_in])
    y += [char_to_int[seq_out]]


new = 'BCD'
new_char = np.array([[char_to_int[new_char]] for new_char in new])
new_char = np.reshape(new_char,(1,3,1))
new_char = new_char / 25
pred = model.predict(new_char, verbose=0)
int_to_char[pred.argmax()]

In [ ]:
#lstm & Gru

import tensorflow_datasets as tfds
import numpy as np
import tensorflow as tf

imdb, info = tfds.load('imdb_reviews', with_info=True, as_supervised=True)
info

train_data, test_data = imdb['train'], imdb['test']
train_data

for d, l in train_data:
    print(d)
    print(l)
    break
-----
training_sentences = []
training_labels = []
testing_sentences = []
testing_labels = []
for d, l in train_data:
    training_sentences.append(str(d.numpy()))
    training_labels.append(l.numpy())

for d, l in test_data:
    testing_sentences.append(str(d.numpy()))
    testing_labels.append(l.numpy())

---
training_labels = np.array(training_labels)
testing_labels = np.array(testing_labels)

from collections import Counter
Counter(training_labels)
Counter(testing_labels)

# Preprocessing the data
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

tokenizer = Tokenizer(num_words=10000)
tokenizer.fit_on_texts(training_sentences)
tokenizer.word_index
tokenizer.fit_on_texts(testing_sentences)
tokenizer.word_index

sequences = tokenizer.texts_to_sequences(training_sentences)
sequences

padded = pad_sequences(sequences, maxlen=500, truncating='post', padding='post')
padded.shape

testing_sequences = tokenizer.texts_to_sequences(testing_sentences)
testing_padded = pad_sequences(testing_sequences, maxlen=500, truncating='post', padding='post')
testing_padded.shape


from keras.models import Sequential
from keras.layers import SimpleRNN, LSTM, Dense, Embedding, Bidirectional
from keras.layers import LSTM, GRU

model_rnn = Sequential([
    Embedding(10000, 50, input_length=500),
    SimpleRNN(32),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
])

history_rnn = model_rnn.fit(padded, training_labels, epochs=10,
                            validation_data = (testing_padded, testing_labels))

import matplotlib.pyplot as plt

plt.xlabel('Accuracy')
plt.ylim(0,1)
plt.grid()
plt.plot(history_rnn.history['accuracy'], label = 'accuracy')
plt.plot(history_rnn.history['val_accuracy'], label = 'val_accuracy')
plt.legend()

model_lstm = Sequential([
    Embedding(10000, 50, input_length=500),
    LSTM(32),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
])
history_lstm = model_lstm.fit(padded, training_labels, epochs=10, 
                              validation_data = (testing_padded, testing_labels))

plt.xlabel('Accuracy')
plt.ylim(0,1)
plt.grid()
plt.plot(history_lstm.history['accuracy'], label = 'accuracy')
plt.plot(history_lstm.history['val_accuracy'], label = 'val_accuracy')
plt.legend()

model_gru = Sequential([
    Embedding(10000, 50, input_length=500),
    GRU(32),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
])
history_gru = model_gru.fit(padded, training_labels, epochs=10,
                            validation_data = (testing_padded, testing_labels))